# Drowning detection: Kaggle training (underwater + above-water)
| View | Dataset | Classes |
|---|---|---|
| `underwater` | [Figshare Underwater Drowning Detection Dataset](https://figshare.com/articles/dataset/Underwater_Drowning_Detection_Dataset/29497235?file=56039351) (CC BY 4.0) | swimming / struggling / drowning |
| `abovewater` | [Roboflow water-rescue-dataset v1](https://universe.roboflow.com/cuizh/water-rescue-dataset/dataset/1) (CC BY 4.0) | 2 classes, mapped to swimming / drowning |

1. Settings → Accelerator **GPU T4 x2 or P100**, Internet **on**.
2. Add-ons → Secrets → add `ROBOFLOW_API_KEY` (Roboflow → Settings → API Keys) and tick it for this notebook. The key never appears in code or output.
3. Run All. Each view trains YOLO11n and YOLO11s; expect roughly 1–2 hours per view. Datasets are downloaded to `/tmp`, so they are not saved as notebook output.
4. Download `results.zip` from the Output panel.

Reported scores from dataset authors (e.g. 97.5% mAP50 underwater) are reference only.

In [ ]:
from pathlib import Path
import subprocess, sys, shutil, json, time, urllib.error, urllib.request, urllib.parse, zipfile
VIEWS = ['underwater', 'abovewater']  # remove one to train only the other
REPO = 'https://github.com/VuongQuocHung/drowining-detect.git'
SOURCE = Path('/tmp/drowining-detect')
CODE = SOURCE/'model_training/pipeline'
WORK = Path('/kaggle/working')
DATA = {
    'underwater': {'root': Path('/tmp/underwater/datasets'), 'rename': {},
                   'source': 'https://figshare.com/articles/dataset/Underwater_Drowning_Detection_Dataset/29497235?file=56039351'},
    'abovewater': {'root': Path('/tmp/abovewater'), 'rename': {},
                   'source': 'https://universe.roboflow.com/cuizh/water-rescue-dataset/dataset/1'},
}
# If audit stops on an unknown class name, map it here, e.g. DATA['abovewater']['rename'] = {'person': 'swimming'}.
# Val/test copies of images identical to another split are excluded and listed in audit.json.
DROP_EXACT_DUPLICATES = True
if not (CODE/'audit.py').exists():
    # Sparse checkout: only the pipeline, not model weights or demo videos.
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse',REPO,str(SOURCE)],check=True)
    subprocess.run(['git','-C',str(SOURCE),'sparse-checkout','set','model_training/pipeline'],check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE/'requirements-kaggle.txt')], check=True)

Restart the Python session after installation if Torch/NumPy were already imported, then rerun the cell above. Save a notebook version with outputs to retain checkpoints.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU accelerator'
import torchvision
print('Python:', sys.version)
print('Torch:', torch.__version__, 'TorchVision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))
boxes = torch.tensor([[0., 0., 1., 1.]], device='cuda')
scores = torch.tensor([1.], device='cuda')
assert torchvision.ops.nms(boxes, scores, 0.5).numel() == 1

In [ ]:
def fetch(url, target):
    def progress(blocks, size, total):
        done = blocks*size
        if total > 0 and (blocks % 2000 == 0 or done >= total):
            print(f'\r  {min(done,total)/1e6:,.0f} / {total/1e6:,.0f} MB', end='', flush=True)
    urllib.request.urlretrieve(url, target, progress)
    print()

def download_underwater(root):
    archive = Path('/tmp/underwater.zip')
    fetch('https://ndownloader.figshare.com/files/56039351', archive)
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(root.parent)  # the zip holds datasets/{images,labels}/{train,val}
    archive.unlink()

def download_abovewater(root):
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
    url = 'https://api.roboflow.com/cuizh/water-rescue-dataset/1/yolov11?' + urllib.parse.urlencode({'api_key': key})
    for _ in range(60):
        try:
            with urllib.request.urlopen(url) as response:
                link = json.load(response).get('export', {}).get('link')
        except urllib.error.HTTPError as exc:
            # The message names the problem (e.g. invalid key) without echoing the key.
            raise RuntimeError(f'Roboflow API {exc.code}: {exc.read().decode()[:300]}') from None
        if link:
            break
        time.sleep(10)  # Roboflow builds the export on the first request
    else:
        raise RuntimeError('Roboflow export was not ready after 10 minutes; rerun this cell')
    archive = Path('/tmp/abovewater.zip')
    fetch(link, archive)
    root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(root)  # {train,valid,test}/{images,labels} + data.yaml
    archive.unlink()

for view in VIEWS:
    root = DATA[view]['root']
    if not root.exists() or not any(root.iterdir()):
        print(f'Downloading {view} dataset...')
        {'underwater': download_underwater, 'abovewater': download_abovewater}[view](root)
    print(view, '->', sorted(p.name for p in root.iterdir()))

In [ ]:
from IPython.display import display, Image
for view in VIEWS:
    config, prepared = DATA[view], WORK/'prepared'/view
    command = [sys.executable, str(CODE/'audit.py'), str(config['root']), '--output', str(prepared), '--source', config['source']]
    for old, new in config['rename'].items():
        command += ['--rename', f'{old}={new}']
    if DROP_EXACT_DUPLICATES:
        command.append('--drop-exact-duplicates')
    subprocess.run(command, check=True)
    report = json.loads((prepared/'audit.json').read_text())
    print(f'=== {view}: classes {report["names"]}')
    print(json.dumps({k: report[k] for k in ('total_images', 'split_counts', 'split_class_boxes', 'limitations')}, indent=2))
    print('Near-duplicate cross-split pairs:', len(report['near_cross_split']),
          '| Dropped exact duplicates:', len(report['dropped_exact_duplicates']))
    display(Image(filename=str(prepared/'annotations.jpg')))

Review annotation previews and audit.json before training. Invalid labels stop preparation. Without source groups the original splits are retained; report that limitation and do not present validation as independent test data.

In [ ]:
# Resume: attach a previous saved notebook output, then copy its experiments/ into WORK/'experiments'.
# shutil.copytree('/kaggle/input/previous-run/experiments', WORK/'experiments', dirs_exist_ok=True)
for view in VIEWS:
    print(f'=== Training {view}')
    subprocess.run([sys.executable, str(CODE/'train.py'), '--data', str(WORK/'prepared'/view/'data.yaml'),
                    '--output', str(WORK/'experiments'/view), '--name', view, '--resume'], check=True)

In [ ]:
selected = {}
for view in VIEWS:
    output = WORK/'experiments'/view
    selected[view] = json.loads((output/'selected.json').read_text())
    print(f'=== {view}')
    for result in json.loads((output/'comparison.json').read_text()):
        print(f"{result['model']}: mAP50={result['map50']:.3f} mAP50-95={result['map50_95']:.3f} "
              f"p50={result['latency_p50_ms']:.1f} ms", json.dumps(result['per_class']))
    print('Selected by validation:', selected[view]['model'])

## Held-out test
Runs once on the checkpoint already chosen by validation, only for views whose data has an independent test split (the Roboflow export has one; the Figshare dataset does not). Test scores never choose weights.

In [ ]:
import yaml
for view in VIEWS:
    data = WORK/'prepared'/view/'data.yaml'
    if 'test' not in yaml.safe_load(data.read_text()):
        print(f'{view}: no test split, validation only')
        continue
    output = WORK/'experiments'/view
    subprocess.run([sys.executable, str(CODE/'train.py'), '--data', str(data), '--output', str(output),
                    '--name', view, '--test-checkpoint', selected[view]['checkpoint']], check=True)
    test = json.loads((output/'test.json').read_text())
    print(f"{view} test: mAP50={test['map50']:.3f} mAP50-95={test['map50_95']:.3f}", json.dumps(test['per_class']))

## Deploy and rollback
`results.zip` contains, per view, `<view>_selected.pt`, selected/comparison/test/environment JSON, audit/manifest and plots; epoch and last checkpoints are left out to keep it small. Copy `underwater_selected.pt` and `abovewater_selected.pt` into `fastapi-backend/model/detection_model/`: the backend uses them automatically and falls back to the original weights when they are removed. No accuracy or soak result is claimed until an actual run completes.

In [ ]:
bundle = WORK/'results.zip'
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as zipped:
    for file in sorted((WORK/'experiments').rglob('*')):
        if file.is_file() and not (file.name.startswith('epoch') or file.name == 'last.pt'):
            zipped.write(file, file.relative_to(WORK))
print(f'{bundle} ({bundle.stat().st_size/1e6:.1f} MB)')